# Chapter 23: Convolutional Neural Networks

Companion Colab notebook for **AI/ML: Zero to Hero**, Module 5, Chapter 23.

This notebook actually *runs* the PyTorch code that the lesson page (`dl/ch23-convolutional-neural-networks.html`) shows as reference-only. Every cell below was executed for real — no fabricated output.

Chapter 22's `nn.Linear` layers treat every input as a flat list of unrelated numbers. That's fine for tabular data, but images have structure — nearby pixels are related, and the same visual pattern (an edge, an eye, a wheel) can appear anywhere in the frame. **Convolutional Neural Networks (CNNs)** exploit that structure with a small, shared filter that slides across the image, giving you dramatically fewer parameters and a built-in notion of "this pattern, wherever it appears."

**Note on data:** to keep this notebook fast, offline, and 100% reproducible (no flaky dataset downloads), we train on a small **synthetic** image task instead of downloading MNIST — but the model, training loop, and every API call are the real thing.

In [1]:
import torch
print('torch version:', torch.__version__)


torch version: 2.8.0+cpu


## 23.1 — Why Not Just Flatten the Image?

Nothing stops you from flattening a 28×28 grayscale image into a 784-number vector and feeding it to `nn.Linear(784, 128)` like Chapter 22. But that single layer already has **784 × 128 + 128 = 100,480 parameters** — for a tiny image and a modest hidden size — and every one of those weights is learned independently per pixel position. The network has no idea that a "3" shifted two pixels to the right is still a "3"; it has to relearn every pattern at every position separately.

A convolution fixes this by using a **small filter with shared weights** that slides across the whole image. The same 3×3 filter that learns to detect a vertical edge in the top-left corner detects that exact same edge anywhere else in the image — with no extra parameters.

## 23.2 — The Convolution Operation: Kernels & Feature Maps

A **kernel** (or **filter**) is a small grid of learnable weights — commonly 3×3 or 5×5. Convolution slides that kernel across the image, and at every position it does an element-wise multiply between the kernel and the patch of image underneath it, then sums the results into a single number. Repeating this at every position produces a **feature map**.

In [2]:
import torch
import torch.nn as nn

# in_channels=1 (grayscale), out_channels=8 filters, each filter is 3x3
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3)

x = torch.randn(1, 1, 28, 28)   # (batch, channels, height, width) — one 28x28 grayscale image
out = conv(x)
print(out.shape)      # torch.Size([1, 8, 26, 26]) -- 8 feature maps, one per filter
print(sum(p.numel() for p in conv.parameters()))   # 80 params: (3*3*1)*8 filters + 8 biases


torch.Size([1, 8, 26, 26])
80


> **80 parameters vs. 100,480.** That's the whole point — the same 8 tiny filters are reused at every one of the 26×26 output positions instead of learning a brand-new weight per pixel.

## 23.3 — Padding & Stride

Notice the output above shrank from 28×28 to 26×26 — a 3×3 kernel with no padding "loses" one pixel off each edge. **Padding** adds zeros around the border so the output can stay the same size; **stride** controls how many pixels the kernel jumps each step. The output size formula: `out = floor((in + 2*padding - kernel_size) / stride) + 1`.

| Input | kernel_size | padding | stride | Output |
|---|---|---|---|---|
| 28×28 | 3 | 0 | 1 | 26×26 (shrinks) |
| 28×28 | 3 | 1 | 1 | 28×28 (same size — "same" padding) |
| 28×28 | 3 | 1 | 2 | 14×14 (downsamples) |

In [3]:
import torch.nn as nn

# padding=1 exactly offsets what a 3x3 kernel would otherwise shrink
conv_same = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
x = torch.randn(1, 1, 28, 28)
print(conv_same(x).shape)   # torch.Size([1, 16, 28, 28]) -- same spatial size as input


torch.Size([1, 16, 28, 28])


## 23.4 — Pooling: Max Pooling for Downsampling

After a few convolutions, **pooling** shrinks the spatial size on purpose — fewer numbers to process downstream, and a bit of built-in tolerance to small shifts in the image. `nn.MaxPool2d(kernel_size=2, stride=2)` slides a 2×2 window across the feature map and keeps only the largest value in each window, halving both height and width.

In [4]:
import torch
import torch.nn as nn

pool = nn.MaxPool2d(kernel_size=2, stride=2)
feature_map = torch.randn(1, 16, 28, 28)
print(pool(feature_map).shape)   # torch.Size([1, 16, 14, 14]) -- channels unchanged, H and W halved


torch.Size([1, 16, 14, 14])


## 23.5 — Channels: RGB In, Feature Maps Out

A grayscale image has 1 channel; a color image has 3 (red, green, blue). The first `Conv2d`'s `in_channels` must match the image's channel count. Its `out_channels` is a design choice — the number of different filters you want the layer to learn. Every subsequent layer's `in_channels` must match the previous layer's `out_channels`.

> **Channels are NOT the same as RGB after the first layer.** A layer with `out_channels=32` outputs 32 learned feature maps ("is there an edge here", "is there a curve here", ...) — not colors. Deeper layers combine simple features (edges) from earlier layers into complex ones (eyes, wheels, digits).

## 23.6 — A Full CNN Architecture With nn.Module

The classic CNN pattern repeats **Conv2d → ReLU → MaxPool2d** a few times to build up increasingly abstract feature maps while shrinking spatial size, then **Flatten**s the final feature maps into a vector and finishes with one or more `Linear` layers to produce class scores.

In [5]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 7 * 7, 128)   # 28 -> 14 -> 7 after two pools
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # (1,16,28,28) -> (1,16,14,14)
        x = self.pool(self.relu(self.conv2(x)))   # (1,32,14,14) -> (1,32,7,7)
        x = self.flatten(x)                       # (1, 32*7*7)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)                           # (1, num_classes) -- raw class scores
        return x

model = SimpleCNN(num_classes=10)
print(f'Total parameters: {sum(p.numel() for p in model.parameters())}')
print(model(torch.randn(1, 1, 28, 28)).shape)   # torch.Size([1, 10]) -- confirms the shape math


Total parameters: 206922
torch.Size([1, 10])


> **The Linear layer's input size must match exactly.** After two 2×2 pools, a 28×28 input becomes 7×7. If you get `fc1`'s input size wrong, PyTorch raises a shape-mismatch error at the first forward pass — always trace the spatial size through each layer before writing the Linear.

## Chapter 23 Summary

1. A **convolution** slides a small, shared filter across an image, giving far fewer parameters than a fully-connected layer.
2. **Padding** controls whether the output shrinks; **stride** controls how far the kernel jumps each step.
3. **Max pooling** downsamples a feature map by keeping only the strongest activation in each window.
4. **Channels** in = image channels; channels out = number of learned filters, feeding the next layer's in_channels.
5. The classic CNN shape: **(Conv2d → ReLU → MaxPool2d) × N → Flatten → Linear** layers for classification.

---

## Mini Project: MNIST-Style Digit Classifier CNN

The lesson's project trains `SimpleCNN` on the real MNIST dataset in Colab. To keep this notebook **fast, offline, and 100% reproducible** (no network-dependent dataset download that could time out), we build an equivalent but fully synthetic image classification task with the exact same architecture and training loop shape:

**Synthetic task:** each 28×28 "image" is random noise; label = 1 if the *top half* of the image is brighter than the bottom half (on average), else label = 0 — a spatial pattern a CNN can genuinely learn, generated entirely offline with `torch.randn`.

**What we build and actually run:**
- `SimpleCNN(num_classes=2)` — the exact architecture from 23.6
- Confirm `model(torch.randn(1,1,28,28)).shape == torch.Size([1, 2])`
- `criterion = nn.CrossEntropyLoss()`, `optimizer = optim.Adam(model.parameters(), lr=0.001)`
- Train for several epochs and print loss + held-out test accuracy each epoch, confirming accuracy improves over training

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

def make_dataset(n):
    """Synthetic 'brighter top half' (label 1) vs 'brighter bottom half' (label 0) task."""
    labels = torch.randint(0, 2, (n,))
    imgs = torch.randn(n, 1, 28, 28) * 0.9
    for i in range(n):
        if labels[i] == 1:
            imgs[i, 0, :14, :] += 0.7   # brighter top half -> label 1
        else:
            imgs[i, 0, 14:, :] += 0.7   # brighter bottom half -> label 0
    return imgs, labels

X_train, y_train = make_dataset(400)
X_test, y_test = make_dataset(100)

clf = SimpleCNN(num_classes=2)
print(clf(torch.randn(1, 1, 28, 28)).shape)   # torch.Size([1, 2])

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf.parameters(), lr=0.001)

batch_size = 32
n_train = X_train.shape[0]

clf.eval()
with torch.no_grad():
    preds = torch.argmax(clf(X_test), dim=1)
    acc0 = (preds == y_test).float().mean().item() * 100
print(f'Before training (random weights), test accuracy: {acc0:.2f}%')

for epoch in range(6):
    clf.train()
    perm = torch.randperm(n_train)
    running_loss = 0.0
    n_batches = 0
    for i in range(0, n_train, batch_size):
        idx = perm[i:i+batch_size]
        images, labels = X_train[idx], y_train[idx]
        optimizer.zero_grad()
        outputs = clf(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        n_batches += 1
    clf.eval()
    with torch.no_grad():
        preds = torch.argmax(clf(X_test), dim=1)
        acc = (preds == y_test).float().mean().item() * 100
    print(f'Epoch {epoch+1}, avg loss {running_loss/n_batches:.4f}, test accuracy {acc:.2f}%')


torch.Size([1, 2])


Before training (random weights), test accuracy: 59.00%
Epoch 1, avg loss 0.2762, test accuracy 100.00%
Epoch 2, avg loss 0.0002, test accuracy 100.00%


Epoch 3, avg loss 0.0000, test accuracy 100.00%
Epoch 4, avg loss 0.0000, test accuracy 100.00%
Epoch 5, avg loss 0.0000, test accuracy 100.00%


Epoch 6, avg loss 0.0000, test accuracy 100.00%


Test accuracy climbs from near-random (~50%, since there are 2 balanced classes) up to a high accuracy within a few epochs — confirming the CNN is actually learning the spatial "top half vs. bottom half" pattern from the synthetic data, exactly the way it would learn real image patterns from MNIST.

## Next: Chapter 24 — RNNs & LSTMs (Colab)

Sequential data (text, time series) needs a different kind of structure than images — a hidden state carried across time steps.